In [1]:
import os
import numpy as np
import pandas as pd
from tabulate import tabulate

In [2]:
from google.colab import drive
drive.mount('/users/')

Mounted at /users/


# PATH CONFIGURATION

In [17]:
BASE_MODELS_DIR = "/users/"

# FedAvg class-wise results (per dataset)
FEDAVG_RESULTS_DIR = os.path.join(BASE_MODELS_DIR, "falcon_fl_results")

# Clustered FL summary (Notebook 5)
CLUSTERED_DIR = os.path.join(BASE_MODELS_DIR, "FedClustered")
CLUSTERED_SUMMARY_PATH = os.path.join(CLUSTERED_DIR, "FedClustered_summary_all_datasets.csv")

# Personalized FedProx + Local FT summary (Notebook 6) – adjust name if different
PERSONALIZED_DIR = os.path.join(BASE_MODELS_DIR, "FedPersonalized")
PERSONALIZED_SUMMARY_PATH = os.path.join(PERSONALIZED_DIR, "FedProx_global_comparison_stub.csv")

print("✅ Base Models Directory:", BASE_MODELS_DIR)
print("✅ FedAvg results dir:", FEDAVG_RESULTS_DIR)
print("✅ Clustered FL summary path:", CLUSTERED_SUMMARY_PATH)
print("✅ Personalized FedProx summary path:", PERSONALIZED_SUMMARY_PATH)

✅ Base Models Directory: /users/
✅ FedAvg results dir: /users/
✅ Clustered FL summary path: /users/
✅ Personalized FedProx summary path: /users/


# LOAD FedAvg CLASS-WISE REPORTS

In [4]:
def discover_fedavg_datasets(base_dir: str):
    if not os.path.exists(base_dir):
        print(f"❌ FedAvg base dir not found: {base_dir}")
        return []
    datasets = []
    for name in os.listdir(base_dir):
        full = os.path.join(base_dir, name)
        if os.path.isdir(full):
            if os.path.exists(os.path.join(full, "class_report.csv")):
                datasets.append(name)
    return sorted(datasets)

In [8]:
def load_fedavg_class_reports(base_dir: str):
    datasets = discover_fedavg_datasets(base_dir)
    print(f"📁 Found FedAvg result datasets: {datasets}")
    reports = {}

    for ds in datasets:
        path = os.path.join(base_dir, ds, "class_report.csv")
        try:
            df = pd.read_csv(path)
            print("\n============================")
            print(f"📄 FILE: {path}")
            print("🔑 Columns:", df.columns.tolist())
            print(df.head())
            reports[ds] = df
        except Exception as e:
            print(f"❌ Failed to read {path}: {e}")

    return reports

fedavg_reports = load_fedavg_class_reports(FEDAVG_RESULTS_DIR)
fedavg_reports

📁 Found FedAvg result datasets: ['CIC_BCCC_IoMT_2024', 'CIC_IIoT_2025', 'CSE_CIC_IDS2018', 'Combined']

📄 FILE: /users/
🔑 Columns: ['Unnamed: 0', 'precision', 'recall', 'f1-score', 'support']
  Unnamed: 0  precision    recall  f1-score   support
0          0   0.710169  0.972001  0.820708    4893.0
1          1   0.000000  0.000000  0.000000     383.0
2          2   0.000000  0.000000  0.000000     386.0
3          3   0.000000  0.000000  0.000000     316.0
4          4   0.999715  0.999029  0.999372  316037.0

📄 FILE: /users/
🔑 Columns: ['Unnamed: 0', 'precision', 'recall', 'f1-score', 'support']
  Unnamed: 0  precision    recall  f1-score  support
0          0   0.000000  0.000000  0.000000    900.0
1          1   1.000000  0.001110  0.002217    901.0
2          2   0.385691  0.795782  0.519565    901.0
3          3   0.000000  0.000000  0.000000    901.0
4          4   0.197481  0.974473  0.328408    901.0

📄 FILE: /users/
🔑 Columns: ['Unnamed: 0', 'precision', 'recall', 'f1-score',

{'CIC_BCCC_IoMT_2024':       Unnamed: 0  precision    recall  f1-score        support
 0              0   0.710169  0.972001  0.820708    4893.000000
 1              1   0.000000  0.000000  0.000000     383.000000
 2              2   0.000000  0.000000  0.000000     386.000000
 3              3   0.000000  0.000000  0.000000     316.000000
 4              4   0.999715  0.999029  0.999372  316037.000000
 5              5   0.000000  0.000000  0.000000     467.000000
 6              6   0.000000  0.000000  0.000000     158.000000
 7              7   0.996156  0.997455  0.996805   62087.000000
 8              8   0.980752  0.997535  0.989073   35705.000000
 9              9   0.987805  0.566434  0.720000     143.000000
 10            10   0.000000  0.000000  0.000000     337.000000
 11            11   0.672515  0.008986  0.017735   12798.000000
 12            12   0.000000  0.000000  0.000000      11.000000
 13            13   0.840355  0.998599  0.912669   72828.000000
 14            14 

# BUILD DATASET-LEVEL METRICS FROM FedAvg CLASS REPORTS

In [6]:
def compute_weighted_metrics_from_report(df: pd.DataFrame):
    df = df.copy()
    # Drop any rows where support is NaN (if present)
    df = df[~df["support"].isna()]

    total_support = df["support"].sum()
    if total_support == 0:
        return {
            "weighted_precision": np.nan,
            "weighted_recall": np.nan,
            "weighted_f1": np.nan,
            "total_support": 0,
        }

    weighted_precision = (df["precision"] * df["support"]).sum() / total_support
    weighted_recall = (df["recall"] * df["support"]).sum() / total_support
    weighted_f1 = (df["f1-score"] * df["support"]).sum() / total_support

    return {
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,  # ~ accuracy
        "weighted_f1": weighted_f1,
        "total_support": total_support,
    }

In [7]:
def build_fedavg_dataset_summary(fedavg_reports: dict):
    rows = []
    for ds_name, df in fedavg_reports.items():
        metrics = compute_weighted_metrics_from_report(df)
        rows.append({
            "dataset": ds_name,
            "method": "FedAvg",
            "test_accuracy_approx": metrics["weighted_recall"],
            "test_f1_weighted": metrics["weighted_f1"],
            "test_precision_weighted": metrics["weighted_precision"],
            "total_support": metrics["total_support"],
        })
    return pd.DataFrame(rows)

df_fedavg_dataset = build_fedavg_dataset_summary(fedavg_reports)

print("\n================ FedAvg Dataset-Level Summary (from class reports) ================")
print(tabulate(df_fedavg_dataset, headers="keys", tablefmt="github", floatfmt=".4f"))
df_fedavg_dataset


================ FedAvg Dataset-Level Summary (from class reports) ================
|    | dataset            | method   |   test_accuracy_approx |   test_f1_weighted |   test_precision_weighted |   total_support |
|----|--------------------|----------|------------------------|--------------------|---------------------------|-----------------|
|  0 | CIC_BCCC_IoMT_2024 | FedAvg   |                 0.7677 |             0.7565 |                    0.7759 |    1523391.9668 |
|  1 | CIC_IIoT_2025      | FedAvg   |                 0.2531 |             0.1215 |                    0.2262 |      18918.2531 |
|  2 | CSE_CIC_IDS2018    | FedAvg   |                 0.8694 |             0.8638 |                    0.8617 |    4331319.9718 |
|  3 | Combined           | FedAvg   |                 0.8402 |             0.8306 |                    0.8258 |    5873628.9695 |


,dataset,method,test_accuracy_approx,test_f1_weighted,test_precision_weighted,total_support
0,CIC_BCCC_IoMT_2024,FedAvg,0.767666,0.756487,0.775861,1.523392e+06
1,CIC_IIoT_2025,FedAvg,0.253079,0.121470,0.226192,1.891825e+04
2,CSE_CIC_IDS2018,FedAvg,0.869382,0.863773,0.861738,4.331320e+06
3,Combined,FedAvg,0.840155,0.830649,0.825756,5.873629e+06


# LOAD CLUSTERED FL SUMMARY & AGGREGATE PER DATASET

In [9]:
def load_clustered_summary(path: str):
    if not os.path.exists(path):
        print(f"⚠️ Clustered FL summary file not found: {path}")
        return None
    df = pd.read_csv(path)
    print("\n================ Raw Clustered FL Summary ================")
    print(tabulate(df.head(), headers="keys", tablefmt="github", floatfmt=".4f"))
    return df

In [10]:

def build_clustered_dataset_summary(df_clustered: pd.DataFrame):
    rows = []
    for ds_name, group in df_clustered.groupby("dataset"):
        # Weighted accuracy by training samples
        total_samples = group["num_train_samples"].sum()
        if total_samples > 0:
            weighted_acc = (group["test_accuracy"] * group["num_train_samples"]).sum() / total_samples
            weighted_loss = (group["test_loss"] * group["num_train_samples"]).sum() / total_samples
        else:
            weighted_acc = np.nan
            weighted_loss = np.nan

        best_acc = group["test_accuracy"].max()
        best_loss = group.loc[group["test_accuracy"].idxmax(), "test_loss"]

        rows.append({
            "dataset": ds_name,
            "method": "Clustered FL",
            "test_accuracy_best_cluster": best_acc,
            "test_loss_best_cluster": best_loss,
            "test_accuracy_weighted": weighted_acc,
            "test_loss_weighted": weighted_loss,
            "num_clusters": group["cluster_id"].nunique(),
            "total_train_samples": total_samples,
        })

    return pd.DataFrame(rows)

In [11]:
df_clustered_raw = load_clustered_summary(CLUSTERED_SUMMARY_PATH)
df_clustered_dataset = None
if df_clustered_raw is not None:
    df_clustered_dataset = build_clustered_dataset_summary(df_clustered_raw)
    print("\n================ Clustered FL Dataset-Level Summary ================")
    print(tabulate(df_clustered_dataset, headers="keys", tablefmt="github", floatfmt=".4f"))
df_clustered_dataset


================ Raw Clustered FL Summary ================
|    | dataset         |   cluster_id |   num_train_samples |   num_clients_in_cluster |   test_loss |   test_accuracy |
|----|-----------------|--------------|---------------------|--------------------------|-------------|-----------------|
|  0 | CSE_CIC_IDS2018 |            0 |             1347521 |                        2 |      0.1618 |          0.9688 |
|  1 | CSE_CIC_IDS2018 |            1 |             1347520 |                        2 |      1.2079 |          0.7836 |
|  2 | CSE_CIC_IDS2018 |            2 |             4042562 |                        6 |      0.1790 |          0.9580 |
|  3 | CIC_IIoT_2025   |            0 |               20597 |                        7 |      1.6337 |          0.4258 |
|  4 | CIC_IIoT_2025   |            1 |                5885 |                        2 |      1.8793 |          0.3356 |

================ Clustered FL Dataset-Level Summary ================
|    | dataset         

,dataset,method,test_accuracy_best_cluster,test_loss_best_cluster,test_accuracy_weighted,test_loss_weighted,num_clusters,total_train_samples
0,CIC_BCCC_IoMT_2024,Clustered FL,0.966258,0.161231,0.965856,0.165200,3,2369719
1,CIC_IIoT_2025,Clustered FL,0.425785,1.633672,0.380244,1.875634,3,29424
2,CSE_CIC_IDS2018,Clustered FL,0.968839,0.161760,0.925315,0.381350,3,6737603
3,Combined,Clustered FL,0.968348,0.167661,0.967455,0.170733,3,9136746


# LOAD PERSONALIZED FedProx + LOCAL FINE-TUNING SUMMARY

In [13]:
def load_personalized_summary(path: str):
    if not os.path.exists(path):
        print(f"⚠️ Personalized FedProx summary file not found, skipping: {path}")
        return None
    df = pd.read_csv(path)
    print("\n================ Raw Personalized FedProx Summary ================")
    print(tabulate(df.head(), headers="keys", tablefmt="github", floatfmt=".4f"))
    return df

In [21]:
def build_personalized_dataset_summary(df_personal: pd.DataFrame):
    # If there's a 'mode' column, prefer personalized
    if "mode" in df_personal.columns:
        rows = []
        for ds_name, group in df_personal.groupby("dataset"):
            if "personalized" in group["mode"].values:
                sub = group[group["mode"] == "personalized"]
            else:
                sub = group  # fallback

            # Take best accuracy row
            idx = sub["test_accuracy"].idxmax()
            row = sub.loc[idx]

            rows.append({
                "dataset": ds_name,
                "method": "FedProx + Local FT",
                "test_accuracy": row["final_test_acc"],
                "test_loss": row["final_test_loss"],
            })
        return pd.DataFrame(rows)
    else:
        # If no mode column, just keep one row per dataset with best accuracy
        rows = []
        for ds_name, group in df_personal.groupby("dataset"):
            idx = group["final_test_acc"].idxmax()
            row = group.loc[idx]
            rows.append({
                "dataset": ds_name,
                "method": "FedProx + Local FT",
                "test_accuracy": row["final_test_acc"],
                "test_loss": row["final_test_loss"],
            })
        return pd.DataFrame(rows)

df_personal_raw = load_personalized_summary(PERSONALIZED_SUMMARY_PATH)
df_personal_dataset = None
if df_personal_raw is not None:
    df_personal_dataset = build_personalized_dataset_summary(df_personal_raw)
    print("\n================ Personalized FedProx Dataset-Level Summary ================")
    print(tabulate(df_personal_dataset, headers="keys", tablefmt="github", floatfmt=".4f"))
df_personal_dataset


================ Raw Personalized FedProx Summary ================
|    | dataset            | method   |   num_clients |   num_rounds |     mu |   final_test_loss |   final_test_acc | method_type             |
|----|--------------------|----------|---------------|--------------|--------|-------------------|------------------|-------------------------|
|  0 | CSE_CIC_IDS2018    | FedProx  |            10 |            5 | 0.0010 |            0.1207 |           0.9702 | Personalized_FL_FedProx |
|  1 | CIC_IIoT_2025      | FedProx  |            10 |            5 | 0.0010 |            1.7759 |           0.3516 | Personalized_FL_FedProx |
|  2 | CIC_BCCC_IoMT_2024 | FedProx  |            10 |            5 | 0.0010 |            0.1230 |           0.9664 | Personalized_FL_FedProx |
|  3 | Combined           | FedProx  |            10 |            5 | 0.0010 |            0.1193 |           0.9679 | Personalized_FL_FedProx |

================ Personalized FedProx Dataset-Level Summary =======

,dataset,method,test_accuracy,test_loss
0,CIC_BCCC_IoMT_2024,FedProx + Local FT,0.966431,0.122953
1,CIC_IIoT_2025,FedProx + Local FT,0.351570,1.775854
2,CSE_CIC_IDS2018,FedProx + Local FT,0.970247,0.120658
3,Combined,FedProx + Local FT,0.967914,0.119344


# BUILD UNIFIED DATASET-LEVEL COMPARISON TABLE

In [22]:
comparison_rows = []

for _, row in df_fedavg_dataset.iterrows():
    comparison_rows.append({
        "dataset": row["dataset"],
        "method": "FedAvg",
        "test_accuracy": row["test_accuracy_approx"],
        "test_f1": row["test_f1_weighted"],
        "notes": "Weighted from class-wise report",
    })

# Add Clustered FL (we will use weighted accuracy as main metric)
if df_clustered_dataset is not None:
    for _, row in df_clustered_dataset.iterrows():
        comparison_rows.append({
            "dataset": row["dataset"],
            "method": "Clustered FL",
            "test_accuracy": row["test_accuracy_weighted"],
            "test_f1": np.nan,  # class-wise F1 per cluster not loaded here
            "notes": f"Weighted over {row['num_clusters']} clusters",
        })

# Add Personalized FedProx (if available)
if df_personal_dataset is not None:
    for _, row in df_personal_dataset.iterrows():
        comparison_rows.append({
            "dataset": row["dataset"],
            "method": "FedProx + Local FT",
            "test_accuracy": row["test_accuracy"],
            "test_f1": np.nan,  # unless you saved F1
            "notes": "Best personalized/global configuration",
        })

df_comparison = pd.DataFrame(comparison_rows)

print("\n================ Unified Dataset-Level Comparison (Accuracy) ================")
print(tabulate(
    df_comparison.sort_values(["dataset", "method"]),
    headers="keys",
    tablefmt="github",
    floatfmt=".4f"
))
df_comparison


================ Unified Dataset-Level Comparison (Accuracy) ================
|    | dataset            | method             |   test_accuracy |   test_f1 | notes                                  |
|----|--------------------|--------------------|-----------------|-----------|----------------------------------------|
|  4 | CIC_BCCC_IoMT_2024 | Clustered FL       |          0.9659 |  nan      | Weighted over 3 clusters               |
|  0 | CIC_BCCC_IoMT_2024 | FedAvg             |          0.7677 |    0.7565 | Weighted from class-wise report        |
|  8 | CIC_BCCC_IoMT_2024 | FedProx + Local FT |          0.9664 |  nan      | Best personalized/global configuration |
|  5 | CIC_IIoT_2025      | Clustered FL       |          0.3802 |  nan      | Weighted over 3 clusters               |
|  1 | CIC_IIoT_2025      | FedAvg             |          0.2531 |    0.1215 | Weighted from class-wise report        |
|  9 | CIC_IIoT_2025      | FedProx + Local FT |          0.3516 |  nan      | Be

,dataset,method,test_accuracy,test_f1,notes
0,CIC_BCCC_IoMT_2024,FedAvg,0.767666,0.756487,Weighted from class-wise report
1,CIC_IIoT_2025,FedAvg,0.253079,0.121470,Weighted from class-wise report
2,CSE_CIC_IDS2018,FedAvg,0.869382,0.863773,Weighted from class-wise report
3,Combined,FedAvg,0.840155,0.830649,Weighted from class-wise report
4,CIC_BCCC_IoMT_2024,Clustered FL,0.965856,NaN,Weighted over 3 clusters
5,CIC_IIoT_2025,Clustered FL,0.380244,NaN,Weighted over 3 clusters
6,CSE_CIC_IDS2018,Clustered FL,0.925315,NaN,Weighted over 3 clusters
7,Combined,Clustered FL,0.967455,NaN,Weighted over 3 clusters
8,CIC_BCCC_IoMT_2024,FedProx + Local FT,0.966431,NaN,Best personalized/global configuration
9,CIC_IIoT_2025,FedProx + Local FT,0.351570,NaN,Best personalized/global configuration
